# Fiche 1 — Préparer son environnement de travail

| | |
|---|---|
| 🎯 **Objectif** | Créer un environnement Python isolé, y installer les bibliothèques du projet et vérifier que le notebook l'utilise bien |
| ⏱️ **Durée** | 30 min |
| 🧰 **Prérequis** | Savoir ouvrir un terminal |
| 🔗 **Dans le projet** | Étape 1 (configurer l'environnement) |

> **L'idée clé.** Chaque projet a **sa propre boîte** de bibliothèques, avec des versions notées noir sur blanc. Sans boîte, un projet qui marche aujourd'hui casse le jour où un autre projet met à jour une bibliothèque commune.

## 1. Pourquoi un environnement virtuel ?

Sans environnement, toutes les bibliothèques s'installent au même endroit, pour tous les projets. Le jour où un
projet a besoin de `numpy 1.x` et un autre de `numpy 2.x`, l'un des deux casse. Un **environnement virtuel** est un
dossier (souvent `.venv`) qui contient un Python et les bibliothèques d'**un seul** projet.

Deux façons de faire, au choix :

| Étape | Avec `venv` + `pip` (fourni avec Python) | Avec `uv` (utilisé dans ce dépôt) |
|---|---|---|
| Créer | `python -m venv .venv` | `uv init` |
| Activer | `source .venv/bin/activate` (Windows : `.venv\Scripts\activate`) | inutile : `uv run …` |
| Installer | `pip install requests pillow matplotlib numpy` | `uv add requests pillow matplotlib numpy` |
| Figer les versions | `pip freeze > requirements.txt` | automatique : `uv.lock` |
| Recréer ailleurs | `pip install -r requirements.txt` | `uv sync --frozen` |

## 2. Quel Python exécute ce notebook ?

La question la plus utile quand un `import` échoue : **quel Python** tourne ici ?

In [1]:
import sys
from pathlib import Path


def court(chemin):
    # Remplace le dossier personnel par ~ : plus lisible, et rien de personnel dans une sortie publiée
    return str(chemin).replace(str(Path.home()), "~")


print("Python utilisé :", court(sys.executable))
print("Version        :", sys.version.split()[0])
print("Dans un environnement virtuel ?", sys.prefix != sys.base_prefix)

Python utilisé : ~/Workspace/Mentoring_OC/AI-Engineer/P01-segmentation-vetements-api/.venv/bin/python
Version        : 3.12.15
Dans un environnement virtuel ? True


## 3. Vérifier les versions installées

L'énoncé demande requests, Pillow, Matplotlib et NumPy. On lit leurs versions **dans** le notebook :

In [2]:
from importlib.metadata import PackageNotFoundError, version

for lib in ["requests", "pillow", "matplotlib", "numpy", "python-dotenv"]:
    try:
        print(f"{lib:<14} {version(lib)}")
    except PackageNotFoundError:
        print(f"{lib:<14} ABSENTE : à installer dans l'environnement du projet")

requests       2.34.2
pillow         13.0.0
matplotlib     3.11.2
numpy          2.5.4
python-dotenv  1.2.4


## ⚠️ Piège : « ça marche dans le terminal, mais pas dans Jupyter »

Tu installes une bibliothèque dans le terminal, puis `import` échoue dans le notebook avec `ModuleNotFoundError`.
Presque toujours, Jupyter utilise **un autre Python** que celui de ton terminal. Démonstration : on demande à un
autre Python (celui du système, s'il existe) s'il connaît `dotenv`.

In [3]:
import os
import shutil
import subprocess

# On cherche un python3 dans le PATH, en sautant le dossier de l'environnement du projet
hors_projet = os.pathsep.join(d for d in os.get_exec_path() if not d.startswith(sys.prefix))
autre_python = shutil.which("python3", path=hors_projet)
for nom, python in [("ce notebook", sys.executable), ("python3 du système", autre_python)]:
    if python is None:
        print(f"{nom:<20} : introuvable")
        continue
    test = subprocess.run([python, "-c", "import dotenv"], capture_output=True)
    print(f"{nom:<20} : {court(python)}\n{'':<22} import dotenv → {'OK' if test.returncode == 0 else 'ModuleNotFoundError'}")

ce notebook          : ~/Workspace/Mentoring_OC/AI-Engineer/P01-segmentation-vetements-api/.venv/bin/python
                       import dotenv → OK
python3 du système   : /usr/bin/python3
                       import dotenv → ModuleNotFoundError


**Lecture.** Deux chemins différents = deux Python différents, chacun avec ses bibliothèques. Le remède : installer
dans l'environnement **du projet**, et lancer Jupyter depuis cet environnement (`uv run jupyter lab`, ou choisir le
noyau `.venv` dans VS Code). Le premier réflexe en cas d'`ImportError` : afficher `sys.executable`.

## 4. Un notebook rangé comme le projet

Le guide du mentor vérifie que le notebook a des **sections qui suivent les étapes** du projet. Un plan qui marche :

1. Environnement (imports, versions) · 2. Token · 3. Données · 4. Appel à l'API · 5. Visualisation · 6. Évaluation ·
7. Validation · 8. Coût · 9. Conclusion

Chaque section commence par une phrase qui dit **ce qu'on cherche**, et finit par une phrase qui dit **ce qu'on a
trouvé**. Les imports sont tous en haut. Pas de cellule qui ne marche que si on a exécuté une cellule plus bas.

### ✍️ À toi de jouer

Écris une cellule qui vérifie que les 4 bibliothèques de l'énoncé sont installées, et qui affiche un message clair (`« requests manque : uv add requests »`) pour chacune de celles qui manquent.

In [4]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
from importlib.metadata import PackageNotFoundError, version

requises = ["requests", "pillow", "matplotlib", "numpy"]
manquantes = []
for lib in requises:
    try:
        version(lib)
    except PackageNotFoundError:
        manquantes.append(lib)
for lib in manquantes:
    print(f"{lib} manque : uv add {lib}")
print("Tout est installé" if not manquantes else f"{len(manquantes)} bibliothèque(s) à installer")
```

**Pourquoi ?** on interroge les métadonnées d'installation **sans importer** la bibliothèque : le test ne plante pas, il explique quoi faire. Attention au nom : on installe `pillow`, mais on importe `PIL`.

</details>

## ✅ À retenir

1. Un **environnement virtuel par projet** : ses bibliothèques ne gênent pas celles des autres.
2. Les versions sont **figées** dans un fichier (`uv.lock` ou `requirements.txt`) pour recréer le même environnement ailleurs.
3. `ModuleNotFoundError` dans Jupyter : d'abord `print(sys.executable)`, puis installer dans **cet** environnement.
4. Le notebook suit les **étapes du projet**, une section par étape, imports en haut.